# CLC – Machine Learning on a Microcontroller


### Abstract
***explain***

### 1. Introduction and Objective:

**Project Goal:** Train a neural network to approximate trigonometric functions (sin, cos, tan) and deploy it on an Arduino Nano 33 BLE Sense Rev2 microcontroller for real-time inference with minimal computational resources.

**Objectives:**
- Generate synthetic training data for sin(x), cos(x), and tan(x) functions
- Build and train a neural network model using TensorFlow/Keras
- Convert the trained model to TensorFlow Lite format optimized for microcontrollers
- Validate accuracy of both Keras and TFLite models
- Deploy the TFLite model to Arduino hardware for edge computing applications

**Who Benefits:**
- IoT developers creating embedded ML applications
- Robotics engineers needing efficient trigonometric calculations on resource-constrained devices
- Students learning edge ML and embedded systems
- Any application requiring low-power, real-time mathematical computations

**Methods Overview:**
- **Mathematical:** Generating labeled datasets using numpy's trigonometric functions (sin, cos, tan)
- **Statistical:** 60/20/20 train/validation/test split with proper normalization
- **Algorithms:** Deep neural network with ReLU activations, Adam optimizer, MSE loss, early stopping
- **Software:** TensorFlow/Keras for training, TFLite converter for optimization, Arduino deployment framework

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Training model for sin, cos, and tan functions...")
print("=" * 50)

### 2. Build the Model

### 2. Build the Model

**Model Architecture Design:**

Our model uses a **4-input architecture** with one-hot encoding to handle multiple trigonometric functions in a single network. The inputs are `[x_value, is_sin, is_cos, is_tan]` where the last three are binary flags indicating which function to compute.

**Step-by-Step Model Building Process:**

**1. Generate Training Data (5000 sample points)**
   - **What:** Create x values uniformly distributed across [-π, π] 
   - **How:** Use `np.linspace(-3.14, 3.14, 5000)` to generate evenly spaced points
   - **Why:** Covers the full range where trig functions exhibit their patterns; uniform sampling ensures model learns across all regions equally

**2. Train All Three Functions with Smart Filtering**
   - **What:** Create labeled samples for sin, cos, and tan with one-hot encoding
   - **How:** 
     - Always train sin and cos on all x values (stable functions)
     - Train tan only where `|tan(x)| < 3` to avoid asymptotes at x ≈ ±π/2
   - **Why:** Tan has vertical asymptotes where it approaches ±∞, which would destabilize training. By filtering to |tan(x)| < 3, we focus on learnable regions while avoiding numerical instability

**3. Convert to NumPy Arrays**
   - **What:** Transform Python lists into NumPy arrays for TensorFlow compatibility
   - **How:** `x = np.array(X_samples)` and `y = np.array(y_samples)`
   - **Why:** TensorFlow requires NumPy arrays for efficient vectorized operations and GPU acceleration

**4. Normalize X Values to [0, 1] Range**
   - **What:** Scale x-values from [-π, π] to [0, 1]
   - **How:** `x_normalized[:, 0] = (x[:, 0] + 3.14) / (2 * 3.14)`
   - **Why:** Neural networks learn faster when inputs are on similar scales. The one-hot flags are already 0/1, so normalizing x to [0, 1] prevents the network from being biased toward larger input values. This leads to faster convergence and better gradient flow during backpropagation.

**5. Split Data: Train (60%) / Validation (20%) / Test (20%)**
   - **What:** Divide dataset into three separate subsets
   - **How:** Use `train_test_split` twice - first 80/20 split, then split the 80% into 75/25 (giving 60/20/20 overall)
   - **Why:** 
     - **Training set:** Used to update model weights
     - **Validation set:** Monitors overfitting during training (for early stopping)
     - **Test set:** Final evaluation on completely unseen data to estimate real-world performance

**6. Configure Model Architecture**
   - **What:** Define a 4-layer neural network with 64→64→32→1 neurons
   - **How:** 
     ```python
     tf.keras.Sequential([
         Dense(64, activation='relu', input_shape=(4,)),  # Hidden layer 1
         Dense(64, activation='relu'),                     # Hidden layer 2
         Dense(32, activation='relu'),                     # Hidden layer 3
         Dense(1)                                          # Output layer
     ])
     ```
   - **Why:** 
     - **ReLU activation:** Prevents vanishing gradients, enables learning complex non-linear patterns
     - **Progressive narrowing (64→64→32→1):** Gradually compresses feature space toward single output
     - **Size justification:** 64 neurons provide enough capacity to learn trigonometric patterns without overfitting
     - **150 epochs with early stopping:** Allows model to converge while preventing overfitting via validation monitoring

In [ ]:
# Generate 5000 points
x_base = np.linspace(-3.14, 3.14, 5000)

# Create training data with one-hot encoding
# Input format: [x_value, is_sin, is_cos, is_tan]
X_samples = []
y_samples = []

# Train all 3 functions
for x_val in x_base:
    # Always train sin and cos
    X_samples.append([x_val, 1, 0, 0])  # sin
    y_samples.append(np.sin(x_val))
    X_samples.append([x_val, 0, 1, 0])  # cos
    y_samples.append(np.cos(x_val))

    # Only train tan in "safe" regions where |tan(x)| < 3
    tan_val = np.tan(x_val)
    if np.abs(tan_val) < 3:
        X_samples.append([x_val, 0, 0, 1])  # tan
        y_samples.append(tan_val)

# Convert to numpy arrays
x = np.array(X_samples)
y = np.array(y_samples)

# Normalize x values to [0, 1] range to match one-hot encoding scale
x_normalized = x.copy()
x_normalized[:, 0] = (x[:, 0] + 3.14) / (2 * 3.14)

# Split into train/validation/test (60/20/20)
x_temp, x_test, y_temp, y_test = train_test_split(x_normalized, y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.25, random_state=42)

print(f"Training samples: {len(x_train)}")
print(f"Validation samples: {len(x_val)}")
print(f"Test samples: {len(x_test)}\n")

# Model configuration
hidden_size = 64
epochs = 150

# Build model: 4 inputs [x_normalized, is_sin, is_cos, is_tan] -> 1 output
model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_size, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(hidden_size, activation='relu'),
    tf.keras.layers.Dense(hidden_size // 2, activation='relu'),
    tf.keras.layers.Dense(1)
])

print(f"Model architecture built with {model.count_params()} parameters")

### 3. Training the Model

**Training Strategy:**

**1. Adam Optimizer**
   - **What:** Adaptive Moment Estimation - an advanced gradient descent variant
   - **How:** Automatically adjusts learning rates for each parameter based on first and second moments of gradients
   - **Why:** 
     - Combines benefits of AdaGrad (adaptive learning rates) and RMSProp (exponential moving averages)
     - Handles sparse gradients well, converges faster than standard SGD
     - Works well out-of-the-box with minimal hyperparameter tuning (default learning rate = 0.001)
     - Particularly effective for non-convex problems like neural networks

**2. Mean Squared Error (MSE) Loss Function**
   - **What:** Measures average squared difference between predictions and true values: `MSE = mean((y_pred - y_true)²)`
   - **How:** Computes prediction error, squares it (penalizes large errors more), averages across all samples
   - **Why:** 
     - Natural choice for regression problems (predicting continuous values like sin(x))
     - Differentiable everywhere (required for gradient descent)
     - Heavily penalizes outliers (large errors contribute quadratically)
     - Directly related to variance - minimizing MSE finds the conditional mean

**3. Early Stopping with Validation Data**
   - **What:** Monitors validation loss during training and stops when it stops improving
   - **How:** 
     - `monitor='val_loss'`: Tracks validation set loss after each epoch
     - `patience=30`: Waits 30 epochs for improvement before stopping
     - `restore_best_weights=True`: Reverts model to best weights found (prevents keeping overfit model)
   - **Why:** 
     - **Prevents overfitting:** Training loss can keep decreasing while validation loss increases (model memorizing training data)
     - **Saves computation:** Automatically stops when more training won't help
     - **Optimal generalization:** Finds the point where model best balances fitting training data vs. generalizing to new data
     - **Patience=30 justification:** Allows model to escape local minima and continue improving, but stops if genuinely plateaued

**Training Process Flow:**
1. Model sees batch of training samples → makes predictions
2. MSE loss computed by comparing predictions to true values
3. Adam optimizer calculates gradients (how to adjust weights to reduce loss)
4. Weights updated to minimize loss
5. After each epoch, validation loss checked
6. If validation loss hasn't improved in 30 epochs → stop training and restore best weights
7. Final model represents optimal trade-off between accuracy and generalization

In [ ]:
model.compile(optimizer='adam', loss='mse')

print(f"Model size: {model.count_params()} parameters")
print(f"Training for up to {epochs} epochs with early stopping...\n")

# Early stopping to prevent overfitting
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True,
    verbose=1
)

# Train with validation data and early stopping
model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=epochs,
    callbacks=[early_stopping],
    verbose=1
)

### 4. Build the Application

**TensorFlow Model Construction and Saving:**

Our application is built around a complete workflow from training to deployment. Here's how the TensorFlow model is constructed and prepared:

**1. Model Compilation**
   - **What:** Configures the learning process by specifying optimizer and loss function
   - **How:** `model.compile(optimizer='adam', loss='mse')`
   - **Why:** Must compile before training to set up the computational graph for backpropagation

**2. Training Execution**
   - **What:** Run the training loop with validation monitoring
   - **How:** `model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=150, callbacks=[early_stopping])`
   - **Why:** 
     - `validation_data` allows monitoring generalization during training
     - `callbacks=[early_stopping]` prevents overfitting by stopping when validation loss plateaus
     - Model learns to map inputs `[x, is_sin, is_cos, is_tan]` → output trigonometric value

**3. Model Evaluation**
   - **What:** Test accuracy on completely unseen test data, separated by function type
   - **How:** 
     ```python
     y_pred = model.predict(x_test).flatten()
     # Separate results for sin, cos, tan using one-hot encoding masks
     for func_idx, name in enumerate(['sin', 'cos', 'tan']):
         mask = x_test[:, func_idx] == 1  # Find samples for this function
         accuracy = np.mean(np.abs(y_test[mask] - y_pred[mask]) < 0.05) * 100
     ```
   - **Why:** 
     - Tolerance of 0.05 means predictions within ±0.05 of true value count as correct
     - Per-function accuracy shows if model struggles with specific functions
     - Test set was never seen during training, so accuracy estimates real-world performance

**4. Model Persistence**
   - **What:** Save trained model to disk for later use and conversion
   - **How:** `model.save('trig_model_all', save_format='tf')`
   - **Why:** 
     - SavedModel format (directory with protobuf files) is TensorFlow's standard format
     - Preserves complete model: architecture, weights, optimizer state, training config
     - Required input for TFLite conversion pipeline
     - Enables sharing models across platforms and continuing training later

**Application Flow Summary:**
```
Raw Data → Preprocessing → Model Architecture → Training → Validation → 
Testing → Saving → [Ready for TFLite Conversion]
```

This trained Keras model serves as the foundation that will be converted to TensorFlow Lite format for Arduino deployment in the next step.

In [ ]:
# Test accuracy - separate by function type
y_pred = model.predict(x_test, verbose=0).flatten()

tolerance = 0.05
names = ['sin', 'cos', 'tan']

print("\n" + "=" * 50)
print("RESULTS")
print("=" * 50)
print(f"Total test samples: {len(x_test)}")
print()

all_accuracies = []

# Evaluate each function separately
for func_idx, name in enumerate(names, start=1):
    # Find samples for this function (check one-hot encoding position)
    mask = x_test[:, func_idx] == 1

    if np.sum(mask) > 0:
        func_y_test = y_test[mask]
        func_y_pred = y_pred[mask]

        errors = np.abs(func_y_test - func_y_pred)
        accuracy = np.mean(errors < tolerance) * 100
        mae = np.mean(errors)
        max_error = np.max(errors)

        all_accuracies.append(accuracy)
        print(f"{name}(x): {accuracy:.1f}% accuracy")
        print(f"  MAE={mae:.4f}, Max Error={max_error:.4f}, Samples={np.sum(mask)}")

if len(names) > 1:
    print(f"\nOverall: {np.mean(all_accuracies):.1f}% accuracy")
print("=" * 50)

# Save model
filename = 'trig_model_all'
model.save(filename, save_format='tf')
print(f"\nModel saved to {filename}")

### 5. Conversion to TF Lite

*explain how model was converted to lite version*

### 6. Test the Application

*explain how we tested tf and tf lite model, show code that visualizes and print result*


In [ ]:
# code goes here as applicable

### 7. Analysis:

*analysis/evaluation of the model*


### 8. Deployment to a Microcontroller:

*show steps to deploy to arduino*


In [ ]:
# code goes here as applicable

### 9.Final Conclusion:

*Summarize the findings and discuss their implications.*

### 10. References

*List all mathematical, statistical, scientific and code references*

# EXTRA


14. **Error Estimation:** 

*Estimate the model's error and discuss potential sources of error.*


In [ ]:
# code goes here as applicable


15. **Model Improvement:** 

*Suggest ways to improve the model based on the analysis.*


In [ ]:
# code goes here as applicable